# Uber NYC Pickups — Exploratory Data Analysis

Guided walkthrough of the **Week 1 mentor session** demo (AI-Assisted Python Programming).

**Dataset:** [FiveThirtyEight Uber TLC April 2014](https://github.com/fivethirtyeight/uber-tlc-foil-response) — NYC Uber pickup locations with timestamps.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn.

**EDA goals:**
1. Descriptive statistics
2. Missing value analysis and context-aware treatment
3. Temporal feature extraction
4. Distribution and box plots
5. Correlation heatmap

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("data/uber-raw-data-apr14.csv")

## 1. Load and inspect

Always start EDA by understanding shape, dtypes, and a few raw rows before aggregating anything.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = ["pickup_time", "lat", "lon", "base"]

print(f"Rows: {len(df):,}")
df.head()

In [ ]:
df.info()
df.describe(include="all")

## 2. Missing values

The session stressed that imputation strategy depends on **data behavior and business context** — not blindly filling with a global mean/median.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(3)
pd.DataFrame({"missing": missing, "pct": missing_pct})

In [ ]:
# For coordinates, dropping rows with missing lat/lon is safer than imputing a fake NYC location.
eda = df.dropna(subset=["lat", "lon"]).copy()

# Parse timestamps; invalid dates become NaT and can be dropped or investigated.
eda["pickup_time"] = pd.to_datetime(eda["pickup_time"], errors="coerce")
invalid_times = eda["pickup_time"].isna().sum()
print(f"Dropped invalid timestamps: {invalid_times:,}")
eda = eda.dropna(subset=["pickup_time"])

print(f"Rows after cleaning: {len(eda):,}")

## 3. Temporal feature extraction

Pickup demand varies by hour and day — extract time features before plotting or modeling.

In [ ]:
eda["hour"] = eda["pickup_time"].dt.hour
eda["day_of_week"] = eda["pickup_time"].dt.day_name()
eda["date"] = eda["pickup_time"].dt.date

eda[["pickup_time", "hour", "day_of_week", "base"]].head()

## 4. Distributions

Look at numeric spread and spot outliers before trusting summary stats alone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(eda["lat"], kde=True, ax=axes[0])
axes[0].set_title("Latitude distribution")

sns.histplot(eda["lon"], kde=True, ax=axes[1])
axes[1].set_title("Longitude distribution")

sns.histplot(eda["hour"], bins=24, discrete=True, ax=axes[2])
axes[2].set_title("Pickups by hour of day")

plt.tight_layout()
plt.show()

## 5. Box plots

Compare pickup hour patterns across Uber bases (TLC dispatch bases in NYC).

In [ ]:
top_bases = eda["base"].value_counts().head(5).index
subset = eda[eda["base"].isin(top_bases)]

plt.figure(figsize=(10, 5))
sns.boxplot(data=subset, x="base", y="hour", palette="Set2")
plt.title("Pickup hour by top 5 Uber bases")
plt.xlabel("Base")
plt.ylabel("Hour of day")
plt.show()

## 6. Correlation heatmap

Correlations among numeric features — useful for spotting redundant variables before modeling.

In [ ]:
numeric = eda[["lat", "lon", "hour"]]
corr = numeric.corr(numeric_only=True)

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation heatmap (numeric features)")
plt.show()

## 7. Daily demand pattern

In [ ]:
daily = eda.groupby("date").size().reset_index(name="pickups")
daily["date"] = pd.to_datetime(daily["date"])

plt.figure(figsize=(12, 4))
sns.lineplot(data=daily, x="date", y="pickups")
plt.title("Daily Uber pickups — April 2014")
plt.xlabel("Date")
plt.ylabel("Pickup count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Hour × day-of-week heatmap

Which hours are busiest on which days? A heatmap makes weekly rhythm visible at a glance.

In [ ]:
DAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

hour_day = (
    eda.groupby(["day_of_week", "hour"])
    .size()
    .reset_index(name="pickups")
)
hour_day["day_of_week"] = pd.Categorical(
    hour_day["day_of_week"], categories=DAY_ORDER, ordered=True
)
heatmap_data = hour_day.pivot(index="day_of_week", columns="hour", values="pickups")

plt.figure(figsize=(14, 5))
sns.heatmap(
    heatmap_data,
    cmap="YlOrRd",
    linewidths=0.3,
    cbar_kws={"label": "Pickup count"},
)
plt.title("Uber pickups by day of week and hour — April 2014")
plt.xlabel("Hour of day")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 9. April vs. other months (Apr–Sep 2014)

Same source repo, six consecutive months. Compare total volume and hourly demand shape.

In [ ]:
def load_uber_month(path: Path) -> pd.DataFrame:
    """Load and clean one monthly Uber TLC CSV."""
    month_df = pd.read_csv(path)
    month_df.columns = ["pickup_time", "lat", "lon", "base"]
    month_df = month_df.dropna(subset=["lat", "lon"]).copy()
    month_df["pickup_time"] = pd.to_datetime(month_df["pickup_time"], errors="coerce")
    month_df = month_df.dropna(subset=["pickup_time"])
    month_df["hour"] = month_df["pickup_time"].dt.hour
    month_df["month"] = month_df["pickup_time"].dt.to_period("M").astype(str)
    return month_df


MONTH_FILES = sorted(Path("data").glob("uber-raw-data-*14.csv"))
monthly_frames = [load_uber_month(path) for path in MONTH_FILES]
all_months = pd.concat(monthly_frames, ignore_index=True)

monthly_summary = (
    all_months.groupby("month")
    .agg(pickups=("pickup_time", "size"), days=("pickup_time", lambda s: s.dt.date.nunique()))
    .assign(avg_pickups_per_day=lambda d: (d["pickups"] / d["days"]).round(0))
    .sort_index()
)
monthly_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(
    data=monthly_summary.reset_index(),
    x="month",
    y="pickups",
    ax=axes[0],
    palette="Blues_d",
)
axes[0].set_title("Total pickups by month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Pickup count")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=monthly_summary.reset_index(),
    x="month",
    y="avg_pickups_per_day",
    ax=axes[1],
    palette="Greens_d",
)
axes[1].set_title("Average pickups per day")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Avg daily pickups")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
hourly_by_month = (
    all_months.groupby(["month", "hour"])
    .size()
    .reset_index(name="pickups")
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=hourly_by_month,
    x="hour",
    y="pickups",
    hue="month",
    marker="o",
    palette="tab10",
)
plt.title("Hourly pickup pattern by month")
plt.xlabel("Hour of day")
plt.ylabel("Pickup count")
plt.xticks(range(0, 24))
plt.legend(title="Month", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 10. Key takeaways (from the session)

1. **EDA first** — most project time should go here; it drives model quality more than tuning.
2. **Context-aware cleaning** — coordinates shouldn't be mean-imputed; timestamps need validation.
3. **Feature engineering from domain logic** — hour/day patterns matter for ride-hailing demand.
4. **AI assists, you verify** — always check column names, aggregations, and plot axes when using an assistant.
5. **Compare across time slices** — monthly totals and hourly curves reveal growth and seasonality (Uber expanded rapidly in NYC during 2014).

**Further ideas:** overlay weekend vs. weekday heatmaps, map pickup density geographically, or pull Jan–Jun 2015 from the repo zip for a longer trend.